# **1. Perkenalan Dataset**

Pada proyek ini, kami menggunakan **Wine Dataset** dari `scikit-learn`. Dataset ini merupakan salah satu dataset benchmark klasik untuk klasifikasi multikelas. 

### Deskripsi Dataset:
- **Jumlah Sampel**: 178 sampel
- **Jumlah Fitur**: 13 fitur numerik (fitur kimia dari analisis wine)
- **Jumlah Kelas**: 3 kelas target (mewakili 3 kultivar wine berbeda di Italia)
- **Fitur-fitur**:
  1. `alcohol` (Kandungan alkohol)
  2. `malic_acid` (Asam malat)
  3. `ash` (Kadar abu)
  4. `alcalinity_of_ash` (Alkalinitas abu)
  5. `magnesium` (Kadar magnesium)
  6. `total_phenols` (Total fenol)
  7. `flavanoids` (Kandungan flavonoid)
  8. `nonflavanoid_phenols` (Fenol non-flavonoid)
  9. `proanthocyanins` (Kandungan proantosianin)
  10. `color_intensity` (Intensitas warna)
  11. `hue` (Rona warna)
  12. `od280/od315_of_diluted_wines` (Uji OD280/OD315 untuk wine encer)
  13. `proline` (Kadar prolin)

Tujuannya adalah mengklasifikasikan kultivar wine berdasarkan fitur-fitur kimia tersebut menggunakan algoritma Machine Learning.

# **2. Import Library**

Mengimpor library yang diperlukan untuk pengolahan data, visualisasi, dan preprocessing.

In [1]:
# Install dependencies jika belum terinstall
# !pip install pandas scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries berhasil diimport!")

Libraries berhasil diimport!


# **3. Memuat Dataset**

Memuat dataset Wine dari library `scikit-learn` ke dalam Pandas DataFrame, lalu menyimpannya sebagai file CSV mentah (`wine_raw.csv`).

In [2]:
# Memuat dataset
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

# Simpan raw data ke root folder Eksperimen_SML_Amanda-Mahendra
df.to_csv('../wine_raw.csv', index=False)

print("Shape dataset:", df.shape)
print("\n5 Data Pertama:")
df.head()

Shape dataset: (178, 14)

5 Data Pertama:


# **4. Exploratory Data Analysis (EDA)**

Melakukan analisis eksplorasi data untuk memahami karakteristik dataset, tipe data, missing values, statistik deskriptif, dan korelasi antar fitur.

In [3]:
print("=== INFORMASI DATASET ===")
print(df.info())

print("\n=== STATISTIK DESKRIPTIF ===")
print(df.describe())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DISTRIBUSI TARGET ===")
print(df['target'].value_counts())

# Visualisasi 1: Distribusi Fitur
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(wine.feature_names):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

# Sembunyikan sisa axes yang tidak terpakai
for j in range(len(wine.feature_names), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('eda_distribution.png')
plt.show()
print("Distribusi fitur berhasil divisualisasikan!")

# Visualisasi 2: Heatmap Korelasi
plt.figure(figsize=(14, 10))
corr_matrix = df.drop('target', axis=1).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap - Wine Dataset')
plt.tight_layout()
plt.savefig('eda_correlation.png')
plt.show()
print("Heatmap korelasi berhasil dibuat!")

# Visualisasi 3: Boxplot per kelas
plt.figure(figsize=(15, 6))
for i, col in enumerate(['alcohol', 'malic_acid', 'ash', 'flavanoids']):
    plt.subplot(1, 4, i+1)
    df.boxplot(column=col, by='target', ax=plt.gca())
    plt.title(col)
plt.suptitle('Distribusi Fitur per Kelas')
plt.tight_layout()
plt.savefig('eda_boxplot.png')
plt.show()

=== INFORMASI DATASET ===
<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    flo

# **5. Data Preprocessing**

Melakukan preprocessing data meliputi:
1. Memisahkan fitur (X) dan target (y).
2. Feature scaling menggunakan StandardScaler.
3. Train-test split (80:20) secara stratifikasi.
4. Menyimpan dataset hasil preprocessing ke file CSV (`wine_preprocessing.csv`).

In [4]:
print("=== TAHAP PREPROCESSING ===")

# Pisahkan fitur dan target
X = df.drop('target', axis=1)
y = df['target']

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

# Cek missing values
print(f"\nMissing values: {X.isnull().sum().sum()}")

# Feature Scaling menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=wine.feature_names)

print("\nSebelum scaling (mean):")
print(X.mean().round(2))
print("\nSetelah scaling (mean, harusnya ~0):")
print(X_scaled_df.mean().round(4))

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Simpan hasil preprocessing
df_preprocessed = pd.DataFrame(X_scaled, columns=wine.feature_names)
df_preprocessed['target'] = y.values
df_preprocessed.to_csv('wine_preprocessing.csv', index=False)

print("\n✅ Dataset hasil preprocessing berhasil disimpan sebagai 'wine_preprocessing.csv'")
print(f"Shape: {df_preprocessed.shape}")
df_preprocessed.head()

=== TAHAP PREPROCESSING ===
Shape X: (178, 13)
Shape y: (178,)

Missing values: 0

Sebelum scaling (mean):
alcohol                          13.00
malic_acid                        2.34
ash                               2.37
alcalinity_of_ash                19.49
magnesium                        99.74
total_phenols                     2.30
flavanoids                        2.03
nonflavanoid_phenols              0.36
proanthocyanins                   1.59
color_intensity                   5.06
hue                               0.96
od280/od315_of_diluted_wines      2.61
proline                         746.89
dtype: float64

Setelah scaling (mean, harusnya ~0):
alcohol                        -0.0
malic_acid                     -0.0
ash                            -0.0
alcalinity_of_ash              -0.0
magnesium                      -0.0
total_phenols                   0.0
flavanoids                     -0.0
nonflavanoid_phenols            0.0
proanthocyanins                -0.0
color_int